# 语义分割

## 介绍

语义分割对图像中的每个像素进行分类，生成与输入空间维度相同的密集预测图。与使用边界框的目标检测不同，它为每个像素分配类别标签（例如建筑、道路、植被、水体）。

在遥感中，语义分割将原始图像转换为专题地图，用于环境监测、城市规划、灾害响应和气候科学。本教程涵盖四个逐步更复杂的应用：

1. **从高分辨率航空图像进行建筑检测**，这是一个说明核心概念的二值分割任务。
2. **地表水制图**跨越四个工作流程：标准RGB图像、多光谱Sentinel-2数据、NAIP航空摄影和传感器无关的预训练模型。
3. **云和云阴影检测**使用传感器无关的预训练模型，说明仅推理分割作为关键预处理步骤。
4. **土地覆盖分类**，一个多类别分割问题，将每个像素分配到13个土地覆盖类别之一。

每个应用遵循一致的模式：获取数据、创建训练瓦片、训练U-Net模型、评估性能、运行推理和可视化结果。

## 学习目标

在本教程结束时，您将能够：

- 解释语义分割与图像分类和目标检测的区别
- 描述编码器-解码器架构以及跳跃连接在保空间细节中的作用
- 比较常见的分割架构（U-Net、DeepLabV3+、FPN、PSPNet）和编码器（ResNet、EfficientNet）
- 解释从ImageNet进行迁移学习如何加速地理空间数据的训练
- 通过切片栅格并将矢量标签栅格化为二值或多类别掩码来准备地理空间训练数据
- 使用`geoai`包训练语义分割模型，支持不同的输入波段配置
- 使用损失曲线、IoU和F1分数评估分割性能
- 使用滑动窗口预测在新图像上运行推理，并将栅格预测转换为矢量多边形
- 将训练好的分割模型发布到Hugging Face Hub并从托管模型运行推理

## 语义分割基础

**深度学习架构**定义了神经网络的整体结构——数据如何流经各层以及如何产生预测。**编码器**将输入图像压缩为紧凑的特征表示，而**解码器**将这些特征重建回原始空间分辨率以产生分割图。

简而言之：

- **架构** = 工厂蓝图（总体设计和数据流）
- **编码器** = 预处理线（将输入压缩为学习到的特征）
- **解码器** = 精加工线（将特征重建为像素级输出）

### 分割架构

编码器-解码器架构是语义分割的标准。**跳跃连接**链接相应的编码器和解码器层，将高分辨率特征图直接转发到解码器，使其能够产生空间精确且语义准确的预测。

[segmentation_models.pytorch](https://github.com/qubvel-org/segmentation_models.pytorch)库提供了一个模块化框架，将架构与编码器分离：

| 架构            | 核心思想                                              | 优势                                                    |
| --------------- | -------------------------------------------------------------- | ------------------------------------------------------- |
| `unet`          | 对称编码器-解码器，每层都有跳跃连接                      | 精细边界描绘；在有限数据下表现良好                        |
| `unetplusplus`  | Dense skip connections (nested U-Net)                          | Better feature fusion than standard U-Net               |
| `deeplabv3`     | Atrous convolutions with ASPP module                           | Multi-scale context capture                             |
| `deeplabv3plus` | Encoder-decoder with atrous separable convolution              | Multi-scale context with improved boundary detail       |
| `fpn`           | Feature pyramid aggregation from multiple encoder levels       | Handles objects at widely varying scales                |
| `pspnet`        | Pyramid pooling module for global scene context                | Strong for scenes requiring broad context               |
| `linknet`       | Lightweight decoder with sum-based skip fusion                 | Fast inference with good accuracy                       |
| `manet`         | Multi-scale attention-guided feature fusion                    | Attention mechanism improves boundary precision         |
| `pan`           | Pyramid attention across multiple scales                       | Hierarchical attention for multi-scale features         |
| `upernet`       | Unified perceptual parsing network                             | Comprehensive multi-scale feature integration           |
| `segformer`     | Vision transformer backbone for segmentation                   | Efficient transformer-based segmentation                |
| `dpt`           | Dense prediction transformer                                   | Fine-grained predictions from vision transformer tokens |

对于大多数地理空间分割任务，**带有预训练ResNet编码器的U-Net**是一个出色的起点。

### 编码器和迁移学习

**编码器**将输入图像压缩为有意义的特征表示。**迁移学习**不是从头开始训练，而是从在ImageNet上预训练的编码器开始，它已经识别了边缘、纹理和模式，这些很好地迁移到遥感图像。

常见的编码器选择包括`resnet34`、`resnet50`、`efficientnet-b0`和`mobilenet_v2`。

### 实际实现

The `geoai` package builds on `segmentation_models.pytorch`, providing high-level functions for the common training and inference pipeline.

## 安装

取消注释以下行以安装所需的包。

In [ ]:
# %pip install "geoai-py[extra]"

## 导入库

首先，导入`geoai`包。

In [ ]:
import geoai

## 从航空图像进行建筑检测

建筑检测是一个二值分割问题（建筑vs背景），清楚地介绍了分割工作流程。我们在高分辨率NAIP航空图像上训练U-Net模型，输入为1米分辨率的3波段RGB。

### 下载示例数据

训练数据集包含来自Overture Maps项目的NAIP图像瓦片和相应的建筑轮廓。

In [ ]:
train_raster_url = "https://data.source.coop/opengeos/geoai/naip_rgb_train.tif"
train_vector_url = (
    "https://data.source.coop/opengeos/geoai/naip_train_buildings.geojson"
)
test_raster_url = "https://data.source.coop/opengeos/geoai/naip_test.tif"

In [ ]:
train_raster_path = geoai.download_file(train_raster_url)
train_vector_path = geoai.download_file(train_vector_url)
test_raster_path = geoai.download_file(test_raster_url)

### 可视化示例数据

在训练前检查数据。

In [ ]:
geoai.print_raster_info(train_raster_path, show_preview=False)

In [ ]:
geoai.view_vector_interactive(train_vector_path, tiles=train_raster_path)

显示保留的测试图像。

In [ ]:
geoai.view_raster(test_raster_path)

### 创建训练瓦片

`export_geotiff_tiles`函数将训练栅格和标签切片成512x512像素的瓦片，步长为256像素，创建重叠补丁。

In [ ]:
out_folder = "buildings"
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_vector_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)

### 训练模型

模型使用U-Net架构，编码器为在ImageNet上预训练的ResNet34。

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
)

### 评估模型

训练曲线揭示模型是否有效学习。训练损失和验证损失之间的差距不断扩大表示过拟合。

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/unet_models/training_history.pth",
    figsize=(15, 5),
)

```text
=== 性能指标摘要 ===
IoU     - 最佳: 0.9012 | 最终: 0.8999
F1      - 最佳: 0.9466 | 最终: 0.9458
Precision - 最佳: 0.9650 | 最终: 0.9605
Recall  - 最佳: 0.9490 | 最终: 0.9326
Val Loss - 最佳: 0.0699 | 最终: 0.0699
===================================
```

### 运行推理

`semantic_segmentation`函数以重叠方式在输入栅格上滑动512x512窗口，将每个补丁通过模型运行，并将预测拼接成无缝输出。我们还生成概率图，存储每个像素的每类置信度。

In [ ]:
masks_path = "naip_buildings_prediction.tif"
probability_path = "naip_test_probability_map.tif"
model_path = f"{out_folder}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
    probability_path=probability_path,
)

### 可视化栅格掩码

将预测叠加在原始图像上进行视觉评估。

In [ ]:
geoai.view_raster(
    masks_path,
    nodata=0,
    colormap="binary",
    basemap=test_raster_path,
)

在这个两类示例中，概率图的波段2对应建筑类别，明亮区域表示高置信度。

In [ ]:
geoai.view_raster(probability_path, indexes=[2], basemap=test_raster_path)

### 向量化预测

`orthogonalize`函数将栅格预测转换为多边形，并将其正则化为干净的直角形状。

In [ ]:
output_vector_path = "naip_buildings_prediction.geojson"
gdf = geoai.orthogonalize(masks_path, output_vector_path, epsilon=2)

### 添加几何属性

计算面积、周长和形状指数用于过滤和分析。

In [ ]:
gdf_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")

### 可视化结果

显示按面积着色的检测到的建筑。

In [ ]:
geoai.view_vector_interactive(gdf_props, column="area_m2", tiles=test_raster_path)

过滤掉非常小的多边形（< 10平方米）以去除噪声。

In [ ]:
gdf_filtered = gdf_props[(gdf_props["area_m2"] > 10)]
geoai.view_vector_interactive(gdf_filtered, column="area_m2", tiles=test_raster_path)

创建分割地图，比较预测轮廓（左）与NAIP图像（右）。

In [ ]:
geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=test_raster_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_path,
)

清除GPU内存缓存。

In [ ]:
geoai.empty_cache()

## 地表水制图

地表水制图对于生态系统监测、洪水风险评估和气候研究至关重要。本节演示四个逐步更高级的工作流程：

1. **非地理参考卫星图像**（RGB）介绍基本工作流程。
2. **Sentinel-2多光谱图像**（6波段）演示额外光谱信息如何改善区分能力。
3. **NAIP航空图像**（4波段，1米分辨率）将工作流程应用于常用的最高分辨率数据。
4. **传感器无关的水分割**使用无需自定义训练的预训练模型。

### 使用非地理参考卫星图像进行水体制图

使用标准图像格式（JPG/PNG）而无需地理坐标是学习核心计算机视觉工作流程的自然起点。

#### 下载示例数据

[水体数据集](https://data.source.coop/opengeos/geoai/waterbody-dataset.zip)包含2,841对图像-掩码，覆盖不同地理区域和水体类型。

**数据集特征：**
- **图像对总数**：2,841个训练示例
- **图像格式**：RGB卫星图像（3通道）
- **掩码格式**：二进制掩码（255=水体，0=背景）
- **可变图像大小**：256x256到1024x1024+像素
- **全球覆盖**：来自不同地理区域和水体类型的样本

In [ ]:
url = "https://data.source.coop/opengeos/geoai/waterbody-dataset.zip"

In [ ]:
out_folder = geoai.download_file(url)
print(f"Downloaded dataset to {out_folder}")

解压后的数据集包含`images`和`masks`文件夹。

#### 训练模型

我们使用U-Net架构，编码器为从ImageNet权重初始化的ResNet34。

关键训练参数：
- `num_channels=3`：RGB输入
- `num_classes=2`：二分类（背景vs水体）
- `batch_size=16`：平衡GPU内存使用和梯度稳定性
- `learning_rate=0.001`：Adam优化器的标准起点
- `val_split=0.2`：保留20%数据用于验证
- `target_size=(512, 512)`：将可变大小图像标准化为统一输入大小

有关可用架构和编码器的完整列表，请参考[segmentation_models.pytorch文档](https://smp.readthedocs.io/en/latest/encoders.html)。

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/masks",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=3,
    num_classes=2,
    batch_size=16,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
    target_size=(512, 512),
    verbose=True,
)

训练在`unet_models`目录中生成多个输出文件：

- `best_model.pth`：验证IoU最高的检查点
- `final_model.pth`：最后一个epoch的检查点
- `training_history.pth`：完整的训练指标，用于分析和绘图
- `training_summary.txt`：配置和结果的人类可读摘要

#### 评估模型

**损失**衡量预测与真实值之间的差异；训练损失和验证损失应一起减少。

**IoU（交并比）**是标准的分割指标，定义为重叠面积除以并集面积，范围从0.0到1.0。

**F1分数**是精确率（预测阳性中正确的比例）和召回率（找到的实际阳性比例）的调和平均数，范围从0.0到1.0。

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/unet_models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

```text
=== 性能指标摘要 ===
IoU     - 最佳: 0.7079 | 最终: 0.7079
F1      - 最佳: 0.8035 | 最终: 0.8035
Precision - 最佳: 0.8555 | 最终: 0.8309
Recall  - 最佳: 0.8246 | 最终: 0.8246
Val Loss - 最佳: 0.3272 | 最终: 0.3440
===================================
```

#### 运行推理 on a Single Image

我们选择单张图像进行演示。更改`index`变量以尝试不同图像。

In [ ]:
index = 3
test_image_path = f"{out_folder}/images/water_body_{index}.jpg"
ground_truth_path = f"{out_folder}/masks/water_body_{index}.jpg"
prediction_path = f"{out_folder}/prediction/water_body_{index}.png"
model_path = f"{out_folder}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=test_image_path,
    output_path=prediction_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=32,
)

并排比较显示原始图像、预测水掩码和真实掩码。

In [ ]:
fig = geoai.plot_prediction_comparison(
    original_image=test_image_path,
    prediction_image=prediction_path,
    ground_truth_image=ground_truth_path,
    titles=["Original", "Prediction", "Ground Truth"],
    figsize=(15, 5),
    save_path=f"{out_folder}/prediction/water_body_{index}_comparison.png",
    show_plot=True,
)

#### 运行推理 on Multiple Images

`semantic_segmentation_batch`函数使用一致的参数处理整个目录的图像。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/waterbody-dataset-sample.zip"

In [ ]:
data_dir = geoai.download_file(url)
print(f"Downloaded dataset to {data_dir}")

设置批量处理的输入和输出目录。

In [ ]:
images_dir = f"{data_dir}/images"
predictions_dir = f"{data_dir}/predictions"

In [ ]:
geoai.semantic_segmentation_batch(
    input_dir=images_dir,
    output_dir=predictions_dir,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=3,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=4,
    quiet=True,
)

清除GPU内存缓存。

In [ ]:
geoai.empty_cache()

### 使用Sentinel-2多光谱图像进行水体制图

**Sentinel-2**提供10-20米分辨率的13个光谱波段。额外的光谱波段，特别是近红外和短波红外，显著改善了水体区分能力，因为水在这些波长处强烈吸收。

本分析中使用的六个光谱波段是：

| 波段            | 波长               | 在水体检测中的作用                              |
| --------------- | ------------------ | ----------------------------------------------- |
| 蓝（490纳米）   | 可见光             | 水体吸收，大气校正                              |
| 绿（560纳米）   | 可见光             | 水体清晰度，植被健康状况                        |
| Red (665 nm)    | Visible            | Land-water contrast                             |
| NIR（842纳米）  | 近红外             | 强烈水体吸收；关键区分器                    |
| SWIR1（1610纳米）| 短波红外           | 水体反射率极低；优秀分隔器                  |
| SWIR2 (2190 nm) | Shortwave infrared | Separates water from wet soil and vegetation    |

#### 下载示例数据

地球表面水体数据集包含Sentinel-2 Level 2A图像，具有6个光谱波段和专家标注的水掩码，训练集和验证集分开。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/dset-s2.zip"
data_dir = geoai.download_file(url, output_path="dset-s2.zip")

数据集组织成四个目录：

- `tra_scene` / `tra_truth`：训练图像和相应的水掩码
- `val_scene` / `val_truth`：独立验证图像和掩码

In [ ]:
images_dir = f"{data_dir}/tra_scene"
masks_dir = f"{data_dir}/tra_truth"
tiles_dir = f"{data_dir}/tiles"

#### 创建训练瓦片

`export_geotiff_tiles_batch`函数一次性处理目录中的所有图像-掩码对。

In [ ]:
result = geoai.export_geotiff_tiles_batch(
    images_folder=images_dir,
    masks_folder=masks_dir,
    output_folder=tiles_dir,
    tile_size=512,
    stride=384,
    quiet=True,
)

#### 训练模型

除了6波段输入的`num_channels=6`外，配置与RGB模型相同。

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{tiles_dir}/images",
    labels_dir=f"{tiles_dir}/masks",
    output_dir=f"{tiles_dir}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=6,
    num_classes=2,
    batch_size=32,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
)

#### 评估模型

Sentinel-2模型实现了0.90的最佳验证IoU，由于额外的光谱波段，明显高于仅RGB模型（0.71）。

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{tiles_dir}/unet_models/training_history.pth",
    figsize=(15, 5),
)

```text
=== 性能指标摘要 ===
IoU     - 最佳: 0.8989 | 最终: 0.8989
F1      - 最佳: 0.9323 | 最终: 0.9323
Precision - 最佳: 0.9500 | 最终: 0.9491
Recall  - 最佳: 0.9453 | 最终: 0.9387
Val Loss - 最佳: 0.0550 | 最终: 0.0550
===================================
```

#### 运行推理 on Validation Data

验证集在训练期间完全保留。

In [ ]:
images_dir = f"{data_dir}/val_scene"
masks_dir = f"{data_dir}/val_truth"
predictions_dir = f"{data_dir}/predictions"
model_path = f"{tiles_dir}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation_batch(
    input_dir=images_dir,
    output_dir=predictions_dir,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=6,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=32,
    quiet=True,
)

#### 可视化结果

使用假彩色合成（波段5-4-3：SWIR1-NIR-红）、模型预测和真实掩码的并排比较。

In [ ]:
image_id = "S2A_L2A_20190318_N0211_R061"
test_image_path = f"{data_dir}/val_scene/{image_id}_6Bands_S2.tif"
ground_truth_path = f"{data_dir}/val_truth/{image_id}_S2_Truth.tif"
prediction_path = f"{data_dir}/predictions/{image_id}_6Bands_S2_mask.tif"
save_path = f"{data_dir}/{image_id}_6Bands_S2_comparison.png"

fig = geoai.plot_prediction_comparison(
    original_image=test_image_path,
    prediction_image=prediction_path,
    ground_truth_image=ground_truth_path,
    titles=["Original", "Prediction", "Ground Truth"],
    figsize=(15, 5),
    save_path=save_path,
    show_plot=True,
    indexes=[5, 4, 3],
    divider=5000,
)

#### 将模型应用于新的Sentinel-2图像

我们下载明尼苏达州上空的Sentinel-2场景，并在其上运行训练好的模型作为泛化检查。

In [ ]:
s2_path = "s2.tif"
url = "https://data.source.coop/opengeos/geoai/s2-minnesota-2025-08-31-subset.tif"
geoai.download_file(url, output_path=s2_path)

In [ ]:
geoai.view_raster(
    s2_path, indexes=[4, 3, 2], vmin=0, vmax=5000, layer_name="Sentinel-2"
)

定义输出路径并运行推理。

In [ ]:
s2_mask = "s2_mask.tif"
model_path = f"{tiles_dir}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=s2_path,
    output_path=s2_mask,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=6,
    num_classes=2,
    window_size=512,
    overlap=256,
    batch_size=32,
)

In [ ]:
geoai.view_raster(
    s2_mask,
    nodata=0,
    colormap="binary",
    layer_name="Water",
    basemap=s2_path,
    basemap_args={"indexes": [4, 3, 2], "vmin": 0, "vmax": 5000},
)

#### 向量化水掩码

将预测的栅格掩码转换为矢量多边形。

In [ ]:
output_vector_path = "s2_water_polygons.geojson"
gdf = geoai.raster_to_vector(
    raster_path=s2_mask,
    output_path=output_vector_path,
    min_area=100,
    simplify_tolerance=None,
)

#### 平滑水体多边形

平滑多边形以获得更自然的边界。

In [ ]:
smoothed_path = "s2_water_smoothed.geojson"
gdf = geoai.smooth_vector(
    gdf,
    smooth_iterations=3,
    output_path=smoothed_path,
)

#### 添加几何属性

计算每个检测到的水体的面积和周长。

In [ ]:
gdf_props = geoai.add_geometric_properties(gdf, area_unit="m2", length_unit="m")
gdf_props.head()

#### 过滤小伪影

移除不太可能是实际水体的小检测区域。

In [ ]:
gdf_filtered = gdf_props[gdf_props["area_m2"] > 100]
print(f"Water bodies detected: {len(gdf_filtered)}")
print(f"Removed {len(gdf_props) - len(gdf_filtered)} small artifacts")

#### 可视化水体多边形

显示按面积着色的检测到的水体多边形。

In [ ]:
geoai.view_vector_interactive(
    gdf_filtered,
    column="area_m2",
    opacity=0.5,
    tiles=s2_path,
    tiles_args={"indexes": [4, 3, 2], "vmin": 0, "vmax": 5000},
)

#### 分割地图比较

并排比较检测到的水体与原始图像。

In [ ]:
geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=s2_path,
    left_args={"style": {"color": "blue", "fillOpacity": 0.3}},
    right_args={"indexes": [4, 3, 2], "vmin": 0, "vmax": 5000},
    basemap=s2_path,
)

#### 水体面积统计

分析水体大小的分布。

In [ ]:
print(gdf_filtered["area_m2"].describe())

#### 保存结果

保存最终的水体多边形。

In [ ]:
gdf_filtered.to_file("s2_water_bodies_final.geojson", driver="GeoJSON")
print(f"Saved {len(gdf_filtered)} water body polygons to s2_water_bodies_final.geojson")

### 使用NAIP航空图像进行水体制图

**NAIP** provides 1-meter resolution aerial photography with four bands (Red, Green, Blue, Near-Infrared). At this resolution, NAIP captures fine-scale features invisible at Sentinel-2's 10-20 meter resolution, such as narrow streams, small ponds, and detailed shoreline geometry.

#### 下载示例数据

训练和测试图像是预处理的NAIP场景，带有相应的栅格化水掩码。

In [ ]:
train_raster_url = "https://data.source.coop/opengeos/geoai/naip_water_train.tif"
train_masks_url = "https://data.source.coop/opengeos/geoai/naip_water_masks.tif"
test_raster_url = "https://data.source.coop/opengeos/geoai/naip_water_test.tif"

In [ ]:
train_raster_path = geoai.download_file(train_raster_url)
train_masks_path = geoai.download_file(train_masks_url)
test_raster_path = geoai.download_file(test_raster_url)

检查训练栅格元数据。

In [ ]:
geoai.print_raster_info(train_raster_path, show_preview=False)

#### 可视化示例数据

将训练水掩码叠加在NAIP图像上，然后显示测试图像。

In [ ]:
geoai.view_raster(train_masks_path, nodata=0, opacity=0.5, basemap=train_raster_path)

In [ ]:
geoai.view_raster(test_raster_path)

#### 创建训练瓦片

将训练栅格和水掩码切片成重叠的512x512瓦片。

In [ ]:
out_folder = "naip"

In [ ]:
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_masks_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)

#### 训练模型

NAIP模型使用`num_channels=4`用于四波段输入（RGB + NIR）。

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=4,
    num_classes=2,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.005,
    val_split=0.2,
)

#### 评估模型

模型实现了0.97的最佳验证IoU，反映了高分辨率NAIP图像的优势。

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

```text
=== 性能指标摘要 ===
IoU     - 最佳: 0.9695 | 最终: 0.9479
F1      - 最佳: 0.9817 | 最终: 0.9658
Precision - 最佳: 0.9934 | 最终: 0.9648
Recall  - 最佳: 0.9822 | 最终: 0.9811
Val Loss - 最佳: 0.0105 | 最终: 0.0173
===================================
```

#### 运行推理

将训练好的模型应用于保留的测试图像。

In [ ]:
masks_path = "naip_water_prediction.tif"
model_path = f"{out_folder}/models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=4,
    num_classes=2,
    window_size=512,
    overlap=128,
    batch_size=32,
)

In [ ]:
geoai.view_raster(
    masks_path,
    nodata=0,
    layer_name="Water",
    basemap=test_raster_path,
)

#### 向量化和分析预测

将栅格掩码转换为矢量多边形并按几何属性过滤。

In [ ]:
output_path = "naip_water_prediction.geojson"
gdf = geoai.raster_to_vector(
    masks_path, output_path, min_area=1000, simplify_tolerance=1
)

计算几何属性并显示检测到的水体。

In [ ]:
gdf = geoai.add_geometric_properties(gdf)
len(gdf)

In [ ]:
geoai.view_vector_interactive(gdf, tiles=test_raster_path)

过滤掉高度拉长的特征（伸长率> 10）以去除道路边缘或阴影等假阳性。

In [ ]:
gdf["elongation"].hist()

In [ ]:
gdf_filtered = gdf[gdf["elongation"] < 10]

In [ ]:
len(gdf_filtered)

#### 可视化结果

显示过滤后的水体多边形和分割地图比较。

In [ ]:
geoai.view_vector_interactive(gdf_filtered, tiles=test_raster_path)

In [ ]:
geoai.create_split_map(
    left_layer=gdf_filtered,
    right_layer=test_raster_path,
    left_args={"style": {"color": "red", "fillOpacity": 0.2}},
    basemap=test_raster_path,
)

### 传感器无关的水分割

[OmniWaterMask](https://github.com/DPIRD-DMA/OmniWaterMask)提供一个预训练的传感器无关模型，结合深度学习、NDWI计算和OpenStreetMap参考数据，用于在多个传感器（Sentinel-2、NAIP、Landsat、PlanetScope、Maxar）上进行稳健的水体检测。

`geoai.segment_water()`函数包装OmniWaterMask，通过[Smoothify](https://github.com/DPIRD-DMA/Smoothify)库内置支持向量化和多边形平滑。

#### Sentinel-2水分割

##### 下载Sentinel-2数据

In [ ]:
s2_url = (
    "https://data.source.coop/opengeos/geoai/S2A-L2A-20190318-N0211-R061-6Bands-S2.tif"
)
s2_path = geoai.download_file(s2_url)

##### 可视化输入数据

使用假彩色合成（NIR、红、绿）查看Sentinel-2场景。

In [ ]:
geoai.view_raster(s2_path, indexes=[4, 3, 2], vmax=3000)

##### 运行水分割

使用`"sentinel2"`波段顺序预设运行`segment_water()`。

In [ ]:
s2_mask_path = geoai.segment_water(
    s2_path,
    band_order="sentinel2",
    output_raster="s2_owm_water_mask.tif",
)

##### 可视化栅格掩码

In [ ]:
geoai.view_raster(
    s2_mask_path,
    nodata=0,
    basemap=s2_path,
    opacity=0.5,
    backend="ipyleaflet",
)

##### 向量化和平滑水体

In [ ]:
s2_gdf = geoai.segment_water(
    s2_path,
    band_order="sentinel2",
    output_raster="s2_owm_water_mask.tif",
    output_vector="s2_owm_water_bodies.geojson",
    smooth=True,
    smooth_iterations=3,
    min_size=100,
)

##### 过滤小伪影

In [ ]:
s2_filtered = s2_gdf[s2_gdf["area_m2"] > 100]
print(f"Water bodies detected: {len(s2_filtered)}")
print(f"Removed {len(s2_gdf) - len(s2_filtered)} small artifacts")

##### 可视化水体多边形

In [ ]:
geoai.view_vector_interactive(
    s2_filtered,
    column="area_m2",
    tiles=s2_path,
    tiles_args={"indexes": [4, 3, 2], "vmax": 3000},
)

##### 分割地图比较

In [ ]:
geoai.create_split_map(
    left_layer=s2_filtered,
    right_layer=s2_path,
    left_args={"style": {"color": "blue", "fillOpacity": 0.3}},
    right_args={"indexes": [4, 3, 2], "vmax": 3000},
    basemap=s2_path,
)

#### NAIP水分割

##### 下载NAIP数据

In [ ]:
naip_url = "https://data.source.coop/opengeos/geoai/naip_water_test_subset.tif"
naip_path = geoai.download_file(naip_url)

##### 可视化NAIP图像

使用假彩色合成（NIR、红、绿）查看NAIP场景。

In [ ]:
geoai.view_raster(naip_path, indexes=[4, 1, 2])

##### 运行水分割

使用`"naip"`波段顺序预设运行`segment_water()`，生成栅格掩码和平滑矢量多边形。

In [ ]:
naip_gdf = geoai.segment_water(
    naip_path,
    band_order="naip",
    output_raster="naip_owm_water_mask.tif",
    output_vector="naip_owm_water_bodies.geojson",
    smooth=True,
    smooth_iterations=3,
    min_size=100,
)

##### 可视化栅格掩码

In [ ]:
geoai.view_raster(
    "naip_owm_water_mask.tif",
    nodata=0,
    basemap=naip_path,
    opacity=0.5,
    backend="ipyleaflet",
)

##### 过滤小伪影

In [ ]:
naip_filtered = naip_gdf[naip_gdf["area_m2"] > 100]
print(f"Water bodies detected: {len(naip_filtered)}")
print(f"Removed {len(naip_gdf) - len(naip_filtered)} small artifacts")

##### 可视化水体多边形

In [ ]:
geoai.view_vector_interactive(
    naip_filtered,
    column="area_m2",
    tiles=naip_path,
)

##### 分割地图比较

In [ ]:
geoai.create_split_map(
    left_layer=naip_filtered,
    right_layer=naip_path,
    left_args={"style": {"color": "blue", "fillOpacity": 0.3}},
    basemap=naip_path,
)

In [ ]:
geoai.empty_cache()

## 云和云阴影检测

云污染是光学遥感中最持久的挑战之一。每个像素被分配到四个类别之一：

| 值 | 类别 |
| ----- | ------------ |
| 0     | 清晰 |
| 1     | 厚云 |
| 2     | 薄云 |
| 3     | 云阴影 |

本节使用GeoAI中的[OmniCloudMask](https://github.com/DPIRD-DMA/OmniCloudMask)集成，这是一个传感器无关模型，只需要红、绿和NIR波段。

### 下载示例数据

我们使用田纳西州诺克斯维尔上空的Sentinel-2 L2A子集。

In [ ]:
url = "https://data.source.coop/opengeos/geoai/S2C-MSIL2A-20250920T162001-subset.tif"
s2_path = geoai.download_file(url)

### 预测云掩码

函数读取红（波段1）、绿（波段2）和NIR（波段4）并对每个像素进行分类。

In [ ]:
pred_path = "cloud_mask.tif"
geoai.predict_cloud_mask_from_raster(
    input_path=s2_path,
    output_path=pred_path,
    red_band=1,
    green_band=2,
    nir_band=4,
    batch_size=4,
    inference_dtype="bf16",
)

### 云统计

像素级统计总结云和阴影覆盖。

In [ ]:
import rasterio as rio
import numpy as np

with rio.open(pred_path) as src:
    pred_array = src.read(1)

stats = geoai.calculate_cloud_statistics(pred_array)
for key, value in stats.items():
    if "percent" in key:
        print(f"{key}: {value:.2f}%")
    else:
        print(f"{key}: {value:,}")

```text
total_pixels: 46,930,758
clear_pixels: 36,797,157
thick_cloud_pixels: 6,014,605
thin_cloud_pixels: 485
shadow_pixels: 4,118,511
clear_percent: 78.41%
cloud_percent: 12.82%
shadow_percent: 8.78%
```

### 可视化结果

显示假彩色合成（NIR/红/绿）和预测的云掩码。

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

with rio.open(s2_path) as src:
    scene_fc = src.read([4, 1, 2]).astype(np.float32)

# Percentile-based contrast stretch per band
for i in range(3):
    band = scene_fc[i]
    p2, p98 = np.percentile(band, (2, 98))
    scene_fc[i] = (band - p2) / (p98 - p2)
scene_fc = np.clip(scene_fc, 0, 1)

cmap = ListedColormap(["green", "white", "gray", "black"])
labels = ["Clear", "Thick Cloud", "Thin Cloud", "Cloud Shadow"]
patches = [
    Patch(facecolor=cmap(i), edgecolor="black", label=labels[i]) for i in range(4)
]

fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(scene_fc.transpose(1, 2, 0))
ax[0].set_title("False Color (NIR/R/G)")
ax[0].set_axis_off()
ax[1].imshow(pred_array, cmap=cmap, vmin=0, vmax=3)
ax[1].set_title("Cloud Mask")
ax[1].legend(handles=patches, loc="upper left", fontsize=12)
ax[1].set_axis_off()
plt.tight_layout()
plt.show()

放大有云覆盖的区域以检查边界细节。

In [ ]:
from scipy import ndimage

cloud_binary = (pred_array == 1) | (pred_array == 2)
labeled, _ = ndimage.label(cloud_binary)
if labeled.max() > 0:
    sizes = ndimage.sum(cloud_binary, labeled, range(1, labeled.max() + 1))
    largest_label = np.argmax(sizes) + 1
    cy, cx = ndimage.center_of_mass(cloud_binary, labeled, largest_label)
    cy, cx = int(cy), int(cx)
else:
    cy, cx = pred_array.shape[0] // 2, pred_array.shape[1] // 2

# Crop a 1000x1000 window around the center
h, w = pred_array.shape
half = 500
r0, r1 = max(0, cy - half), min(h, cy + half)
c0, c1 = max(0, cx - half), min(w, cx + half)

fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(scene_fc[:, r0:r1, c0:c1].transpose(1, 2, 0))
ax[0].contour(pred_array[r0:r1, c0:c1], levels=[0.5, 1.5, 2.5], colors="cyan")
ax[0].set_title("False Color with Cloud Contours")
ax[0].set_axis_off()
ax[1].imshow(pred_array[r0:r1, c0:c1], cmap=cmap, vmin=0, vmax=3)
ax[1].set_title("Cloud Mask (Zoomed)")
ax[1].legend(handles=patches, loc="upper left", fontsize=12)
ax[1].set_axis_off()
plt.tight_layout()
plt.show()

### 后处理

清理掩码，转换为矢量多边形，并平滑边界。

In [ ]:
cleaned_mask_path = "cleaned_mask.tif"
geoai.clean_raster(pred_path, cleaned_mask_path)

In [ ]:
cloud_vector = "cloud_vector.geojson"
cloud_gdf = geoai.raster_to_vector(cleaned_mask_path, cloud_vector)

In [ ]:
smoothed_cloud_gdf = geoai.smooth_vector(
    cloud_gdf, smooth_iterations=3, output_path="smoothed_cloud_vector.geojson"
)

### 交互式可视化

将平滑的云多边形叠加在卫星图像上。

In [ ]:
geoai.view_vector_interactive(
    smoothed_cloud_gdf, tiles=s2_path, tiles_args={"indexes": [4, 1, 2], "vmax": 4000}
)

分割地图比较云掩码叠加与原始图像。

In [ ]:
geoai.create_split_map(
    left_layer=smoothed_cloud_gdf,
    right_layer=s2_path,
    left_args={"style": {"color": "cyan", "fillOpacity": 0.2}},
    right_args={"indexes": [4, 1, 2], "vmax": 4000},
    basemap=s2_path,
)

### 几何属性

计算每个云多边形的几何属性。

In [ ]:
props_gdf = geoai.add_geometric_properties(cloud_gdf)
props_gdf.describe()

### 无云掩码

创建可用（无云）像素的二进制掩码。

In [ ]:
cloud_free = geoai.create_cloud_free_mask(pred_array)
usable_pct = cloud_free.sum() / cloud_free.size * 100
print(f"Usable (cloud-free) pixels: {usable_pct:.2f}%")

fig, ax = plt.subplots(1, 2, figsize=(20, 10))
ax[0].imshow(scene_fc.transpose(1, 2, 0))
ax[0].set_title("False Color (NIR/R/G)")
ax[0].set_axis_off()
ax[1].imshow(cloud_free, cmap="RdYlGn", vmin=0, vmax=1)
ax[1].set_title("Cloud-Free Mask (green = usable)")
ax[1].set_axis_off()
plt.tight_layout()
plt.show()

清除GPU内存缓存。

In [ ]:
geoai.empty_cache()

## 土地覆盖分类

**Land cover classification** extends binary segmentation to **multi-class segmentation**, where every pixel is assigned to one of many categories. The key differences are: `num_classes` increases (13 in this example), the loss function and metrics operate across all classes, and class imbalance may require weighted losses or oversampling.

分类方案遵循[Chesapeake Land Cover](https://planetarycomputer.microsoft.com/dataset/chesapeake-lc-13)项目，具有13个土地覆盖类别。训练数据由NAIP 4波段航空图像与栅格化土地覆盖标签配对组成。

### 下载示例数据

训练数据集包含一个NAIP图像瓦片及其相应的13类土地覆盖标签栅格。

In [ ]:
train_raster_url = (
    "https://data.source.coop/opengeos/geoai/m_3807511_ne_18_060_20181104.tif"
)
train_landcover_url = (
    "https://data.source.coop/opengeos/geoai/m_3807511_ne_18_060_20181104_landcover.tif"
)
test_raster_url = (
    "https://data.source.coop/opengeos/geoai/m_3807511_se_18_060_20181104.tif"
)

In [ ]:
train_raster_path = geoai.download_file(train_raster_url)
train_landcover_path = geoai.download_file(train_landcover_url)
test_raster_path = geoai.download_file(test_raster_url)

### 可视化示例数据

可视化叠加在NAIP图像上的训练标签，然后创建分割地图。

In [ ]:
legend_args = {"builtin_legend": "Chesapeake", "title": "Land Cover Type"}
geoai.view_raster(
    train_landcover_path, basemap=train_raster_path, legend_args=legend_args
)

In [ ]:
geoai.create_split_map(
    left_layer=train_landcover_path,
    right_layer=train_raster_path,
)

显示测试图像。

In [ ]:
geoai.view_raster(test_raster_path)

### 创建训练瓦片

标签图像包含整数类别值（0到12）。

In [ ]:
out_folder = "landcover"

In [ ]:
tiles = geoai.export_geotiff_tiles(
    in_raster=train_raster_path,
    out_folder=out_folder,
    in_class_data=train_landcover_path,
    tile_size=512,
    stride=256,
    buffer_radius=0,
)

### 训练模型

配置使用`num_classes=13`用于完整的土地覆盖方案。

In [ ]:
geoai.train_segmentation_model(
    images_dir=f"{out_folder}/images",
    labels_dir=f"{out_folder}/labels",
    output_dir=f"{out_folder}/unet_models",
    architecture="unet",
    encoder_name="resnet34",
    encoder_weights="imagenet",
    num_channels=4,
    num_classes=13,
    batch_size=8,
    num_epochs=20,
    learning_rate=0.001,
    val_split=0.2,
    verbose=True,
    plot_curves=True,
)

### 评估模型

对于多类别分割，检查验证IoU和损失是否继续改善，以及聚合指标是否隐藏了少数类别的弱性能。

In [ ]:
geoai.plot_performance_metrics(
    history_path=f"{out_folder}/unet_models/training_history.pth",
    figsize=(15, 5),
    verbose=True,
)

### 运行推理

将模型应用于训练集中没有的相邻NAIP瓦片。

In [ ]:
masks_path = "landcover_prediction.tif"
model_path = f"{out_folder}/unet_models/best_model.pth"

In [ ]:
geoai.semantic_segmentation(
    input_path=test_raster_path,
    output_path=masks_path,
    model_path=model_path,
    architecture="unet",
    encoder_name="resnet34",
    num_channels=4,
    num_classes=13,
    window_size=512,
    overlap=128,
    batch_size=4,
)

### 可视化结果

使用与训练标签相同的方案为预测类别图着色。

In [ ]:
geoai.write_colormap(masks_path, train_landcover_path, output=masks_path)

In [ ]:
geoai.view_raster(masks_path, basemap=test_raster_path, legend_args=legend_args)

## 发布和重用模型

通过[Hugging Face Hub](https://huggingface.co/docs/hub)共享训练好的模型，让协作者无需访问原始训练数据或计算资源即可运行推理。

### 推送到Hugging Face Hub

您需要免费的Hugging Face账户和写入访问令牌。

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

上传训练好的模型权重和配置。将`"your-username"`替换为您的Hugging Face用户名。

In [ ]:
repo_url = geoai.push_timm_model_to_hub(
    model_path=model_path,
    repo_id="your-username/chesapeake-landcover-unet-resnet34",
    encoder_name="resnet34",
    architecture="unet",
    num_channels=4,
    num_classes=13,
)
print(repo_url)

### 运行推理 from Hub

`timm_segmentation_from_hub`函数自动下载并运行模型——无需本地检查点。

In [ ]:
hub_output = "landcover_hub_prediction.tif"
geoai.timm_segmentation_from_hub(
    input_path=test_raster_path,
    output_path=hub_output,
    repo_id="your-username/chesapeake-landcover-unet-resnet34",
    window_size=512,
    overlap=128,
    batch_size=4,
)

应用相同的颜色映射并可视化以确认结果匹配。

In [ ]:
geoai.write_colormap(hub_output, train_landcover_path, output=hub_output)
geoai.view_raster(hub_output, basemap=test_raster_path, legend_args=legend_args)

清除GPU内存缓存。

In [ ]:
geoai.empty_cache()

## 关键要点

1. 语义分割为每个像素分配类别标签，在输入分辨率下生成密集预测图。

2. 带有跳跃连接的编码器-解码器架构是现代分割模型的基础。

3. 从ImageNet迁移学习通过提供预训练的特征提取器加速训练，这些提取器很好地迁移到遥感领域。

4. 相同的工作流程适用于不同传感器；主要调整是更改`num_channels`。

5. 额外的光谱波段（特别是NIR和SWIR）显著改善了水体等目标的区分能力。

6. IoU和F1分数是标准的分割评估指标，应始终在保留数据上计算。

7. 后处理（向量化、正则化、几何过滤）将栅格掩码转换为GIS就绪的矢量特征。

8. 多类别分割通过增加`num_classes`扩展二值分割，可能需要类别平衡策略。

9. Publishing models to Hugging Face Hub enables sharing and reuse without local training or data access.